In [1]:
!pip -q install folium pandas

In [2]:
import pandas as pd
import numpy as np
import folium
from IPython.display import display

data = {
    "asset_id": [f"AST-{i:03d}" for i in range(1, 16)],
    "asset_name": [
        "Lampu Jalan A", "Rambu Jalan A", "Tempat Sampah A",
        "Lampu Jalan B", "Rambu Jalan B", "Lampu Jalan C",
        "Tempat Sampah B", "Lampu Jalan D", "Rambu Jalan C",
        "Lampu Jalan E", "Tempat Sampah C", "Rambu Jalan D",
        "Lampu Jalan F", "Tempat Sampah D", "Lampu Jalan G"
    ],
    "asset_type": [
        "Lampu Jalan", "Rambu", "Tempat Sampah",
        "Lampu Jalan", "Rambu", "Lampu Jalan",
        "Tempat Sampah", "Lampu Jalan", "Rambu",
        "Lampu Jalan", "Tempat Sampah", "Rambu",
        "Lampu Jalan", "Tempat Sampah", "Lampu Jalan"
    ],
    "latitude": [
        -3.790, -3.795, -3.800, -3.785, -3.810,
        -3.805, -3.780, -3.815, -3.775, -3.820,
        -3.792, -3.802, -3.788, -3.812, -3.798
    ],
    "longitude": [
        102.260, 102.270, 102.250, 102.280, 102.265,
        102.245, 102.275, 102.255, 102.285, 102.240,
        102.290, 102.235, 102.268, 102.282, 102.248
    ],
    "condition": [
        "Baik", "Rusak", "Baik", "Perlu Perbaikan", "Baik",
        "Rusak", "Perlu Perbaikan", "Baik", "Rusak", "Baik",
        "Perlu Perbaikan", "Baik", "Rusak", "Baik", "Perlu Perbaikan"
    ],
    "age_years": [2, 8, 1, 7, 3, 9, 6, 2, 10, 4, 5, 3, 8, 2, 7],
    "importance": [5, 4, 2, 5, 4, 5, 2, 5, 4, 5, 2, 4, 5, 2, 5]
}

df = pd.DataFrame(data)

print("Jumlah aset:", len(df))
display(df.head())

Jumlah aset: 15


,asset_id,asset_name,asset_type,latitude,longitude,condition,age_years,importance
0,AST-001,Lampu Jalan A,Lampu Jalan,-3.790,102.260,Baik,2,5
1,AST-002,Rambu Jalan A,Rambu,-3.795,102.270,Rusak,8,4
2,AST-003,Tempat Sampah A,Tempat Sampah,-3.800,102.250,Baik,1,2
3,AST-004,Lampu Jalan B,Lampu Jalan,-3.785,102.280,Perlu Perbaikan,7,5
4,AST-005,Rambu Jalan B,Rambu,-3.810,102.265,Baik,3,4


In [3]:
# Nilai kondisi: makin tinggi berarti makin bermasalah
condition_score = {
    "Baik": 0,
    "Perlu Perbaikan": 60,
    "Rusak": 100
}

df["condition_score"] = df["condition"].map(condition_score)

# Normalisasi usia menjadi skala 0–100
df["age_score"] = (df["age_years"] / 10).clip(0, 1) * 100

# Normalisasi kepentingan: 1–5 menjadi 0–100
df["importance_score"] = (df["importance"] / 5) * 100

# Rumus skor prioritas
df["priority_score"] = (
    0.60 * df["condition_score"] +
    0.25 * df["age_score"] +
    0.15 * df["importance_score"]
).round(2)

def determine_priority(score):
    if score >= 70:
        return "Tinggi"
    elif score >= 40:
        return "Sedang"
    return "Rendah"

df["priority"] = df["priority_score"].apply(determine_priority)

# Urutkan aset dari prioritas tertinggi
df = df.sort_values("priority_score", ascending=False).reset_index(drop=True)

display(df[
    ["asset_id", "asset_name", "condition",
     "age_years", "priority_score", "priority"]
])

,asset_id,asset_name,condition,age_years,priority_score,priority
0,AST-006,Lampu Jalan C,Rusak,9,97.5,Tinggi
1,AST-009,Rambu Jalan C,Rusak,10,97.0,Tinggi
2,AST-013,Lampu Jalan F,Rusak,8,95.0,Tinggi
3,AST-002,Rambu Jalan A,Rusak,8,92.0,Tinggi
4,AST-015,Lampu Jalan G,Perlu Perbaikan,7,68.5,Sedang
5,AST-004,Lampu Jalan B,Perlu Perbaikan,7,68.5,Sedang
6,AST-007,Tempat Sampah B,Perlu Perbaikan,6,57.0,Sedang
7,AST-011,Tempat Sampah C,Perlu Perbaikan,5,54.5,Sedang
8,AST-010,Lampu Jalan E,Baik,4,25.0,Rendah
9,AST-001,Lampu Jalan A,Baik,2,20.0,Rendah


In [4]:
color_map = {
    "Tinggi": "red",
    "Sedang": "orange",
    "Rendah": "green"
}

# Pusat peta pada koordinat data demonstrasi
asset_map = folium.Map(
    location=[df["latitude"].mean(), df["longitude"].mean()],
    zoom_start=14,
    tiles="OpenStreetMap"
)

for _, row in df.iterrows():
    popup_text = f"""
    <b>{row['asset_name']}</b><br>
    ID: {row['asset_id']}<br>
    Jenis: {row['asset_type']}<br>
    Kondisi: {row['condition']}<br>
    Usia: {row['age_years']} tahun<br>
    Skor prioritas: {row['priority_score']}<br>
    Prioritas: {row['priority']}
    """

    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=8,
        color=color_map[row["priority"]],
        fill=True,
        fill_color=color_map[row["priority"]],
        fill_opacity=0.8,
        popup=folium.Popup(popup_text, max_width=300),
        tooltip=row["asset_name"]
    ).add_to(asset_map)

asset_map

In [5]:
print("=== RINGKASAN GEOASSET ===")
print("Total aset:", len(df))
print("Aset rusak:", (df["condition"] == "Rusak").sum())
print("Perlu perbaikan:",
      (df["condition"] == "Perlu Perbaikan").sum())
print("Prioritas tinggi:", (df["priority"] == "Tinggi").sum())

print("\nJumlah aset berdasarkan prioritas:")
display(
    df["priority"]
    .value_counts()
    .reindex(["Tinggi", "Sedang", "Rendah"], fill_value=0)
    .rename_axis("Prioritas")
    .reset_index(name="Jumlah Aset")
)

print("\nLima aset dengan prioritas tertinggi:")
display(df[
    ["asset_id", "asset_name", "condition", "priority_score"]
].head(5))

=== RINGKASAN GEOASSET ===
Total aset: 15
Aset rusak: 4
Perlu perbaikan: 4
Prioritas tinggi: 4

Jumlah aset berdasarkan prioritas:


,Prioritas,Jumlah Aset
0,Tinggi,4
1,Sedang,4
2,Rendah,7



Lima aset dengan prioritas tertinggi:


,asset_id,asset_name,condition,priority_score
0,AST-006,Lampu Jalan C,Rusak,97.5
1,AST-009,Rambu Jalan C,Rusak,97.0
2,AST-013,Lampu Jalan F,Rusak,95.0
3,AST-002,Rambu Jalan A,Rusak,92.0
4,AST-015,Lampu Jalan G,Perlu Perbaikan,68.5


In [6]:
# Ekspor dataset hasil analisis
df.to_csv("geoasset_results.csv", index=False)

# Simpan peta sebagai file HTML interaktif
asset_map.save("geoasset_map.html")

# Simpan ringkasan jumlah aset per prioritas
summary = (
    df.groupby("priority")
    .size()
    .reindex(["Tinggi", "Sedang", "Rendah"], fill_value=0)
    .rename("jumlah_aset")
    .reset_index()
)
summary.to_csv("geoasset_summary.csv", index=False)

print("File berhasil disiapkan:")
print("- geoasset_results.csv")
print("- geoasset_map.html")
print("- geoasset_summary.csv")

File berhasil disiapkan:
- geoasset_results.csv
- geoasset_map.html
- geoasset_summary.csv


In [7]:
from google.colab import files

files.download("geoasset_results.csv")
files.download("geoasset_map.html")
files.download("geoasset_summary.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>